# Inverse Prediction — Find Input Combinations from Output Constraints

Uses trained linear (RidgeCV) models for all 6 targets combined with `scipy.optimize.minimize` (SLSQP)
to find input combinations that satisfy user-defined output constraints.

**Approach**: Multi-start optimization with random initial guesses. The objective is flat (0.0) so the
solver focuses entirely on satisfying the inequality constraints from all 6 models.

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV


class LinearModel:
    """Must match training notebook class for joblib deserialization."""
    def __init__(self, linear_features):
        self.linear_features = linear_features
        self.baseline = None

    def fit(self, X, y):
        self.baseline = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]).fit(X[self.linear_features], y)
        return self

    def predict(self, X):
        return self.baseline.predict(X[self.linear_features])

### Load config and training data

In [ ]:
with open("config.json") as f:
    config = json.load(f)

data_dir = "../forward_prediction"
model_dir = "../models_linear"

train_X = pd.read_csv(f"{data_dir}/train.csv")
print(f"Train: {train_X.shape}")
print(f"Constraints: {config['constraints']}")
print(f"Target solutions: {config['n_solutions']}")

### Feature engineering helpers (same as forward predict)

In [ ]:
_EPS = 1e-6

def build_geometry_base(df):
    X = df.copy()
    X["L_char"] = (X["energy"] / (X["atmosphere"] * X["gravity"])) ** 0.25
    X["pi_strength"] = X["strength"] / (X["atmosphere"] * X["gravity"] * X["L_char"])
    X["sin_angle"] = np.sin(X["angle_rad"])
    X["cos_angle"] = np.cos(X["angle_rad"])
    return X

def build_geometry_fines(df):
    X = build_geometry_base(df)
    X["D_fines"] = 40.0
    X["pi_threshold_fines"] = X["D_fines"] / X["L_char"]
    return X

def build_geometry_oversize(df):
    X = build_geometry_base(df)
    X["D_oversize"] = 120.0
    X["pi_threshold_oversize"] = X["D_oversize"] / X["L_char"]
    return X

In [ ]:
_LOG_SOURCES_BASE = {
    "pi_strength": "log_pi_strength",
    "coupling": "log_coupling",
    "porosity": "log_porosity",
    "shape_factor": "log_shape",
    "energy": "log_energy",
    "strength": "log_strength",
    "gravity": "log_gravity",
    "atmosphere": "log_atmosphere",
}

_LOG_SOURCES_FINES = {**_LOG_SOURCES_BASE, "pi_threshold_fines": "log_pi_threshold_fines"}
_LOG_SOURCES_OVERSIZE = {**_LOG_SOURCES_BASE, "pi_threshold_oversize": "log_pi_threshold_oversize"}

In [ ]:
class ZLogBase:
    def __init__(self, log_sources):
        self._log_sources = log_sources
        self._scalers = {}

    def _raw_logs(self, geom):
        logs = pd.DataFrame(index=geom.index)
        for source, log_col in self._log_sources.items():
            logs[log_col] = np.log(geom[source].clip(lower=_EPS))
        return logs

    def _z_transform(self, geom):
        X = geom.copy()
        logs = self._raw_logs(geom)
        for log_col in logs:
            raw_log = logs[[log_col]].to_numpy()
            X[f"{log_col}_raw"] = raw_log.ravel()
            X[log_col] = self._scalers[log_col].transform(raw_log).ravel()
        return X, logs

    def _fit_base(self, geom):
        logs = self._raw_logs(geom)
        for log_col in logs:
            self._scalers[log_col] = StandardScaler().fit(logs[[log_col]].to_numpy())
        return logs


class ZLogP80(ZLogBase):
    def __init__(self):
        super().__init__(_LOG_SOURCES_BASE)

    def fit(self, geom):
        self._fit_base(geom)
        return self

    def transform(self, geom):
        X, logs = self._z_transform(geom)
        X["log_pi_strength_sq"] = X["log_pi_strength"] ** 2
        X["log_pi_strength_cu"] = X["log_pi_strength"] ** 3
        X["log_energy_x_cos_angle"] = X["log_energy"] * X["cos_angle"]
        X["log_energy_x_sin_angle"] = X["log_energy"] * X["sin_angle"]
        X["log_coupling_sq"] = X["log_coupling"] ** 2
        X["log_porosity_sq"] = X["log_porosity"] ** 2
        return X


class ZLogR95(ZLogBase):
    def __init__(self):
        super().__init__(_LOG_SOURCES_BASE)
        self._sq_sources = {"log_porosity": "log_porosity_sq", "log_coupling": "log_coupling_sq"}

    def fit(self, geom):
        logs = self._fit_base(geom)
        for log_col, sq_col in self._sq_sources.items():
            sq_vals = logs[log_col].to_numpy().reshape(-1, 1) ** 2
            self._scalers[sq_col] = StandardScaler().fit(sq_vals)
        return self

    def transform(self, geom):
        X, logs = self._z_transform(geom)
        for log_col, sq_col in self._sq_sources.items():
            sq_vals = logs[log_col].to_numpy().reshape(-1, 1) ** 2
            X[sq_col] = self._scalers[sq_col].transform(sq_vals).ravel()
        X["log_energy_x_cos_angle"] = X["log_energy"] * X["cos_angle"]
        X["log_atmosphere_x_cos_angle"] = X["log_atmosphere"] * X["cos_angle"]
        X["log_strength_x_cos_angle"] = X["log_strength"] * X["cos_angle"]
        return X


class ZLogDerived(ZLogBase):
    def __init__(self, log_sources, derived_sources):
        super().__init__(log_sources)
        self._derived_sources = derived_sources

    def _compute_derived(self, X, logs):
        raw = {}
        for name, spec in self._derived_sources.items():
            src, fn = spec
            if isinstance(src, tuple):
                a = logs[src[0]].to_numpy() if src[0] in logs.columns else X[src[0]].to_numpy()
                b = logs[src[1]].to_numpy() if src[1] in logs.columns else X[src[1]].to_numpy()
                raw[name] = fn(a, b)
            else:
                vals = logs[src].to_numpy() if src in logs.columns else X[src].to_numpy()
                raw[name] = fn(vals)
        return raw

    def fit(self, geom):
        logs = self._fit_base(geom)
        X_tmp = geom.copy()
        X_tmp["sin_angle"] = np.sin(X_tmp["angle_rad"])
        X_tmp["cos_angle"] = np.cos(X_tmp["angle_rad"])
        derived = self._compute_derived(X_tmp, logs)
        for name, vals in derived.items():
            self._scalers[name] = StandardScaler().fit(vals.reshape(-1, 1))
        return self

    def transform(self, geom):
        X, logs = self._z_transform(geom)
        derived = self._compute_derived(X, logs)
        for name, vals in derived.items():
            X[name] = self._scalers[name].transform(vals.reshape(-1, 1)).ravel()
        return X

In [ ]:
# ---------- Derived feature specs ----------

DERIVED_R50_FINES = {
    "log_porosity_sq": ("log_porosity", lambda x: x ** 2),
    "log_coupling_sq": ("log_coupling", lambda x: x ** 2),
    "cos_angle_sq": ("cos_angle", lambda x: x ** 2),
    "log_shape_sq": ("log_shape", lambda x: x ** 2),
    "log_shape_cu": ("log_shape", lambda x: x ** 3),
    "log_energy_cu": ("log_energy", lambda x: x ** 3),
    "log_atm_x_sin": (("log_atmosphere", "sin_angle"), lambda a, b: a * b),
    "log_strength_x_cos": (("log_strength", "cos_angle"), lambda a, b: a * b),
    "log_energy_x_gravity": (("log_energy", "log_gravity"), lambda a, b: a * b),
}

DERIVED_R50_OVERSIZE = {
    "log_coupling_sq": ("log_coupling", lambda x: x ** 2),
    "log_porosity_sq": ("log_porosity", lambda x: x ** 2),
    "log_pi_strength_sq": ("log_pi_strength", lambda x: x ** 2),
    "log_pi_strength_cu": ("log_pi_strength", lambda x: x ** 3),
    "log_shape_sq": ("log_shape", lambda x: x ** 2),
    "log_shape_cu": ("log_shape", lambda x: x ** 3),
    "log_energy_x_cos": (("log_energy", "cos_angle"), lambda a, b: a * b),
    "log_strength_x_cos": (("log_strength", "cos_angle"), lambda a, b: a * b),
    "log_atm_x_sin": (("log_atmosphere", "sin_angle"), lambda a, b: a * b),
}

DERIVED_FINES_FRAC = {
    "log_coupling_sq": ("log_coupling", lambda x: x ** 2),
    "log_porosity_sq": ("log_porosity", lambda x: x ** 2),
    "log_pi_threshold_fines_sq": ("log_pi_threshold_fines", lambda x: x ** 2),
    "log_shape_sq": ("log_shape", lambda x: x ** 2),
    "log_shape_cu": ("log_shape", lambda x: x ** 3),
    "log_pi_strength_sq": ("log_pi_strength", lambda x: x ** 2),
    "log_pi_strength_cu": ("log_pi_strength", lambda x: x ** 3),
    "log_energy_x_coupling": (("log_energy", "log_coupling"), lambda a, b: a * b),
    "log_strength_x_coupling": (("log_strength", "log_coupling"), lambda a, b: a * b),
    "log_atm_x_sin": (("log_atmosphere", "sin_angle"), lambda a, b: a * b),
}

DERIVED_OVERSIZE_FRAC = {
    "log_pi_threshold_oversize_sq": ("log_pi_threshold_oversize", lambda x: x ** 2),
    "log_pi_threshold_oversize_cu": ("log_pi_threshold_oversize", lambda x: x ** 3),
    "log_pi_strength_sq": ("log_pi_strength", lambda x: x ** 2),
    "log_pi_strength_cu": ("log_pi_strength", lambda x: x ** 3),
    "log_shape_sq": ("log_shape", lambda x: x ** 2),
    "log_shape_cu": ("log_shape", lambda x: x ** 3),
    "log_coupling_sq": ("log_coupling", lambda x: x ** 2),
    "log_porosity_sq": ("log_porosity", lambda x: x ** 2),
    "log_energy_x_strength": (("log_energy", "log_strength"), lambda a, b: a * b),
    "log_atm_x_strength": (("log_atmosphere", "log_strength"), lambda a, b: a * b),
    "log_strength_x_coupling": (("log_strength", "log_coupling"), lambda a, b: a * b),
    "log_porosity_x_strength": (("log_porosity", "log_strength"), lambda a, b: a * b),
    "log_atm_x_cos": (("log_atmosphere", "cos_angle"), lambda a, b: a * b),
}

### Load models and fit z-log transformers on training data

In [ ]:
TARGET_CONFIGS = {
    "P80": {
        "build_geometry": build_geometry_base,
        "z_log_factory": lambda: ZLogP80(),
        "model_file": "model_P80.joblib",
        "transform": "length",
    },
    "R95": {
        "build_geometry": build_geometry_base,
        "z_log_factory": lambda: ZLogR95(),
        "model_file": "model_R95.joblib",
        "transform": "length",
    },
    "R50_fines": {
        "build_geometry": build_geometry_base,
        "z_log_factory": lambda: ZLogDerived(_LOG_SOURCES_BASE, DERIVED_R50_FINES),
        "model_file": "model_R50_fines.joblib",
        "transform": "length",
    },
    "R50_oversize": {
        "build_geometry": build_geometry_base,
        "z_log_factory": lambda: ZLogDerived(_LOG_SOURCES_BASE, DERIVED_R50_OVERSIZE),
        "model_file": "model_R50_oversize.joblib",
        "transform": "length",
    },
    "fines_frac": {
        "build_geometry": build_geometry_fines,
        "z_log_factory": lambda: ZLogDerived(_LOG_SOURCES_FINES, DERIVED_FINES_FRAC),
        "model_file": "model_fines_frac.joblib",
        "transform": "logit",
    },
    "oversize_frac": {
        "build_geometry": build_geometry_oversize,
        "z_log_factory": lambda: ZLogDerived(_LOG_SOURCES_OVERSIZE, DERIVED_OVERSIZE_FRAC),
        "model_file": "model_oversize_frac.joblib",
        "transform": "logit",
    },
}

# Fit z-log transformers on training data and load models
models = {}
z_logs = {}
target_names = list(TARGET_CONFIGS.keys())

for target_name, cfg in TARGET_CONFIGS.items():
    train_geom = cfg["build_geometry"](train_X)
    z_log = cfg["z_log_factory"]().fit(train_geom)
    z_logs[target_name] = z_log
    models[target_name] = joblib.load(f"{model_dir}/{cfg['model_file']}")
    print(f"Loaded model for {target_name}")

print(f"\nAll {len(models)} models loaded.")

### Define forward prediction function and constraints

In [ ]:
# Input order: energy, angle_rad, coupling, strength, porosity, gravity, atmosphere, shape_factor
INPUT_COLUMNS = ["energy", "angle_rad", "coupling", "strength", "porosity", "gravity", "atmosphere", "shape_factor"]
RAW_COLUMNS = ["porosity", "atmosphere", "gravity", "coupling", "strength", "shape_factor", "energy", "angle_rad"]

# Build bounds from config
b = config["input_bounds"]
bounds = tuple(
    (b[col]["min"], b[col]["max"]) for col in INPUT_COLUMNS
)

print("Input bounds:")
for col, (lo, hi) in zip(INPUT_COLUMNS, bounds):
    print(f"  {col}: [{lo}, {hi}]")

In [ ]:
def predict_all_6_models(inputs):
    """Run all 6 forward models on a single input vector.
    
    Args:
        inputs: array of 8 values in INPUT_COLUMNS order
                [energy, angle_rad, coupling, strength, porosity, gravity, atmosphere, shape_factor]
    Returns:
        dict mapping target name -> predicted value (in original scale)
    """
    row = {col: [inputs[i]] for i, col in enumerate(INPUT_COLUMNS)}
    df = pd.DataFrame(row, columns=RAW_COLUMNS)
    
    results = {}
    for target_name, cfg in TARGET_CONFIGS.items():
        geom = cfg["build_geometry"](df)
        X = z_logs[target_name].transform(geom)
        y_transformed = models[target_name].predict(X)[0]
        
        if cfg["transform"] == "length":
            results[target_name] = np.exp(y_transformed) * X["L_char"].to_numpy()[0]
        else:
            results[target_name] = expit(y_transformed)
    
    return results


# Cache to avoid redundant model calls within a single optimizer step.
# Each constraint function is called separately by SLSQP, but they all
# need the same set of predictions for the same input vector.
_pred_cache = {}

def _get_predictions(inputs):
    key = tuple(np.round(inputs, 10))
    if key not in _pred_cache:
        _pred_cache.clear()
        _pred_cache[key] = predict_all_6_models(inputs)
    return _pred_cache[key]

In [ ]:
# Flat objective: just find feasible points
def objective_function(inputs):
    return 0.0

# Build constraint functions from config (using cache for efficiency)
cons = config["constraints"]

def make_min_constraint(target_name, min_val):
    """result >= min_val  =>  result - min_val >= 0"""
    def constraint_fn(inputs):
        return _get_predictions(inputs)[target_name] - min_val
    return constraint_fn

def make_max_constraint(target_name, max_val):
    """result <= max_val  =>  max_val - result >= 0"""
    def constraint_fn(inputs):
        return max_val - _get_predictions(inputs)[target_name]
    return constraint_fn

scipy_constraints = []
for target_name, limits in cons.items():
    if "min" in limits:
        scipy_constraints.append({"type": "ineq", "fun": make_min_constraint(target_name, limits["min"])})
    if "max" in limits:
        scipy_constraints.append({"type": "ineq", "fun": make_max_constraint(target_name, limits["max"])})

scipy_constraints = tuple(scipy_constraints)
print(f"Total constraints: {len(scipy_constraints)} (min + max for each of {len(cons)} targets)")
print()
for target_name, limits in cons.items():
    print(f"  {target_name}: [{limits.get('min', '-inf')}, {limits.get('max', 'inf')}]")

### Multi-start optimization engine

In [ ]:
n_solutions = config["n_solutions"]
max_attempts = config["max_attempts"]

valid_combinations = []
valid_predictions = []
attempts = 0

np.random.seed(42)

print(f"Hunting for {n_solutions} valid combinations across 6 models...")

while len(valid_combinations) < n_solutions and attempts < max_attempts:
    attempts += 1
    
    # Generate a random starting guess within allowed bounds
    random_guess = [np.random.uniform(lo, hi) for (lo, hi) in bounds]
    
    # Run the solver
    result = minimize(
        objective_function,
        x0=random_guess,
        method='SLSQP',
        bounds=bounds,
        constraints=scipy_constraints,
        options={'maxiter': 5000, 'disp': False}
    )
    
    if result.success:
        new_solution = np.round(result.x, 4)
        
        # Check for duplicates
        is_duplicate = False
        for saved_solution in valid_combinations:
            if np.allclose(new_solution, saved_solution, atol=1e-3):
                is_duplicate = True
                break
        
        if not is_duplicate:
            valid_combinations.append(new_solution)
            preds = predict_all_6_models(result.x)
            valid_predictions.append(preds)
            print(f"[{len(valid_combinations)}/{n_solutions}] Found at attempt {attempts}  |  "
                  f"P80={preds['P80']:.1f}  fines_frac={preds['fines_frac']:.4f}  "
                  f"oversize_frac={preds['oversize_frac']:.4f}")

# Summary
if len(valid_combinations) == n_solutions:
    print(f"\nSuccessfully generated {n_solutions} unique valid combinations in {attempts} attempts!")
else:
    print(f"\nStopped after {max_attempts} attempts. Only found {len(valid_combinations)} valid combinations.")
    print("Your constraints might be too tight, causing the models to conflict.")

### Output results

In [ ]:
import os

# Build inputs DataFrame
df_inputs = pd.DataFrame(valid_combinations, columns=INPUT_COLUMNS)

# Build predictions DataFrame
df_preds = pd.DataFrame(valid_predictions)

# Combine into one table
df_results = pd.concat([df_inputs, df_preds], axis=1)
df_results.index.name = "solution_id"

# Save to inverse_predict folder
output_dir = "../inverse_predict"
os.makedirs(output_dir, exist_ok=True)

df_results.to_csv(f"{output_dir}/inverse_predictions.csv")
print(f"Saved {len(df_results)} solutions to {output_dir}/inverse_predictions.csv")
print()
print(df_results.to_string())

### Verify: check all solutions satisfy constraints

In [ ]:
violations = 0
for i, preds in enumerate(valid_predictions):
    for target_name, limits in cons.items():
        val = preds[target_name]
        if "min" in limits and val < limits["min"]:
            print(f"  Solution {i}: {target_name}={val:.4f} < min={limits['min']}")
            violations += 1
        if "max" in limits and val > limits["max"]:
            print(f"  Solution {i}: {target_name}={val:.4f} > max={limits['max']}")
            violations += 1

if violations == 0:
    print(f"All {len(valid_predictions)} solutions satisfy all constraints!")
else:
    print(f"\n{violations} constraint violations found.")